# Memory
- Long - Term Memory
- Short - Term Memory

## 환경설정

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
os.getenv("OPENAI_API_KEY")

In [ ]:
os.getenv("TAVILY_API_KEY")

## Tavily 검색 테스트
pip install -U langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch

websearch = TavilySearch(
    max_results = 5,
    topic = "general",
)

tool.invoke("랭체인에 대해서 설명해줘.")

## 그래프 구성
- State 만들기
- Node 만들기
- Node 사이 edge 연결
      - 도구 노드 추가
- conditional Edge
- 컴파일

In [ ]:
from typing import Annotated

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
# 도구

tools = [websearch]

# State

class ChatState(TypedDict):
    messages:Annotated[list, add_messages]

In [ ]:
# LLM
llm = ChatOpenAI(
    model = "gpt-5-nano",
    temperature = 0
)

llm_with_tool = llm.bind_tools(tools)

# Node
def chatbot(state: ChatState):
    messages = state["messages"]
    response = llm_with_tool.invoke(messages)
    return {"messages": [response]} 

# Graph
workflow = StateGraph(ChatState)
workflow.add_node("chatbot",chatbot)

workflow.add_node("tools",ToolNode(tools=tools))

workflow.add_edge(START, "chatbot")
workflow.add_conditional_edges(
    "chatbot", 
    tools_condition,
    # 이부분은 생략 가능!
    {
        "tools": "tools",
        END : END
    }
)
workflow.add_edge("tools","chatbot")
workflow.add_edge("chatbot",END)
agent = workflow.compile()

In [ ]:
from IPython.display import display, Markdown

# Get the Mermaid code
mermaid_code = agent.get_graph().draw_mermaid()

# Display in Jupyter
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
agent

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
user_input = {
    "messages":[
        HumanMessage(content="안녕 내 이름은 제임스야.")
    ]
}

response = agent.invoke(user_input)

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

### 새로운 Thread -> 이전 대화를 기억 X

In [ ]:
user_input = {
    "messages":[
        HumanMessage(content="내 이름이 뭐지? 내이름은 말해봐")
    ]
}

response = agent.invoke(user_input)

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

In [ ]:
from langchain_core.messages import AIMessage

In [ ]:
user_input = {
    "messages":[
        HumanMessage(content="내 이름은 왈도야"),
        AIMessage(content="반가워요 왈도."),
        HumanMessage(content="내 이름이 뭐지? 내이름은 말해봐")
    ]
}

response = agent.invoke(user_input)

for msg in response["messages"]:
    msg.pretty_print()

## 메모리 추가

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent = workflow.compile(checkpointer=memory)

In [ ]:
from IPython.display import display, Markdown

# Get the Mermaid code
mermaid_code = agent.get_graph().draw_mermaid()

# Display in Jupyter
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

agent

In [ ]:
config = {"configurable":{"thread_id":"1"}} # Agent는 이 Thread 번호에 따라서 기억을 가져간다.

user_input = {
    "messages":[
        HumanMessage(content="내 이름은 왈도야"),
        AIMessage(content="반가워요 왈도."),
        HumanMessage(content="내 이름이 뭐지? 내이름은 말해봐")
    ]
}

response = agent.invoke(user_input, config=config)
for msg in response["messages"]:
    msg.pretty_print()

# 에이전트 생성 함수

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent


model = init_chat_model(model="gpt-40", model_provider="openai",temperature = 0)
agent = create_agent(model, [websearch])

In [ ]:
from IPython.display import display, Markdown

# Get the Mermaid code
mermaid_code = agent.get_graph().draw_mermaid()

# Display in Jupyter
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))